# Бектест на прибыльность

**Шаг 7 плана.** Загрузка лучшей модели, бектест с комиссией 0.1%, варианты: торговать на каждом баре и с фильтром по confidence. Сравнение с наивной стратегией (target).

## 1. Загрузка модели и данных

**Важно:** Модель обучена в **07** (шаг 6 плана) с temporal split (8 дней train, 1 val, 1 test). Бектест только на **последнем дне** (test_date) — иначе in-sample завышает доходность.

**Данные:** В 07 для обучения используются только строки с target ∈ {BUY, SELL} (HOLD отфильтрован). Здесь бектест идёт по **полному дню** (все бары с BUY+SELL+HOLD) — так мы проверяем модель в условиях, близких к проду; утечки «очищенные данные везде» нет.

In [5]:
import sys
import os
import numpy as np
import pandas as pd
import joblib

_root = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) in ('01_data_prep','02_targets','03_features','04_models','05_experiments') else os.getcwd()
if _root not in sys.path:
    sys.path.insert(0, _root)

loaded = joblib.load(os.path.join(_root, 'models', 'candidate_baseline_tp_sl_1_05.joblib'))
model = loaded['model']
scaler = loaded['scaler']
FEATURES = loaded['features']
TARGET_COL = loaded.get('target', 'target')
BEST_THRESHOLD = float(loaded.get('threshold', 0.6))

df = pd.read_parquet(os.path.join(_root, 'outputs', 'data_labeled_tp_sl_1_05.parquet'))
sort_col = 'datetime' if 'datetime' in df.columns else 'timestamp'

# valid_full: честный бэктест на ВСЕХ данных (BUY+SELL+HOLD) - как в проде
valid_full = df.dropna(subset=FEATURES + [TARGET_COL]).copy()
valid_full = valid_full[valid_full[TARGET_COL].isin([-1.0, 0.0, 1.0])]
valid_full = valid_full.sort_values(['session_key', sort_col]).reset_index(drop=True)
valid_full['ret_next'] = valid_full.groupby('session_key')['close_price'].pct_change().shift(-1)
valid_full = valid_full.dropna(subset=['ret_next'])
valid_full['date'] = pd.to_datetime(valid_full['datetime'], utc=True).dt.date

dates = sorted(valid_full['date'].unique())
test_date = dates[-1]
artifact_test_date = loaded.get('split', {}).get('test_date')
if artifact_test_date is not None:
    if str(test_date) != str(artifact_test_date):
        import warnings
        warnings.warn(f'Тестовый день в данных ({test_date}) не совпадает с test_date из артефакта 07 ({artifact_test_date}). Проверьте пайплайн.')
else:
    import warnings
    warnings.warn('В артефакте модели нет split.test_date — убедитесь, что модель сохранена из 07 с temporal split.')
print(f'Бектест на том же дне, что test в 07: {test_date}')

bt_full = valid_full[valid_full['date'] == test_date].copy()
bt_full = bt_full.sort_values(['session_key', sort_col]).reset_index(drop=True)

X_bt = scaler.transform(bt_full[FEATURES].fillna(0))
proba = model.predict_proba(X_bt)[:, 1]
bt = bt_full

COMMISSION_RT = 0.001  # 0.1% round-trip (0.05% one-way)
print(f'Модель: {loaded.get("model_name", "unknown")}, val_auc={loaded.get("val_auc", np.nan):.4f}, test_auc={loaded.get("test_auc", np.nan):.4f}')
print(f'Target: {TARGET_COL}, tuned threshold: {BEST_THRESHOLD:.2f}')
print(f'Бектест: полные данные ({test_date}, {len(bt):,} баров, BUY+SELL+HOLD)')

Бектест на том же дне, что test в 07: 2026-02-10
Модель: XGBoost, val_auc=0.7573, test_auc=0.6259
Target: target, tuned threshold: 0.45
Бектест: полные данные (2026-02-10, 26,364 баров, BUY+SELL+HOLD)


### Честный бэктест на полных данных

Данные: пайплайн NB 01–03. Полный набор фичей (22), строки по session_key+datetime, target BUY+SELL+HOLD. Бэктест на всех барах последнего дня.

## 2. Функция бектеста

In [6]:
def _bt(pos, ret, sess, cr):
    pos_prev = np.roll(pos, 1); pos_prev[0] = 0.0
    if sess is not None:
        chg = np.zeros(len(pos), dtype=bool); chg[1:] = sess[1:] != sess[:-1]
        pos_prev = np.where(chg, 0.0, pos_prev)
    nch = int(((pos != pos_prev) & ((pos != 0) | (pos_prev != 0))).sum())
    gross = (pos * ret).sum() * 100
    net = gross - nch * (cr / 2) * 100
    return {'trades': nch, 'gross_%': gross, 'net_%': net, 'avg_net_per_trade': net/nch if nch else np.nan}

def backtest_exit_at_05(proba, ret, cr=0.001, thr=0.5, sess=None):
    pred = np.where(proba >= thr, 1, np.where(proba <= 1-thr, 0, -1))
    n, pos, prev = len(proba), np.zeros(len(proba)), 0.0
    for i in range(n):
        if sess is not None and i and sess[i] != sess[i-1]: prev = 0.0
        if pred[i]==1: pos[i], prev = 1.0, 1.0
        elif pred[i]==0: pos[i], prev = -1.0, -1.0
        else: pos[i] = 0.0
    return _bt(pos, ret, sess, cr)

def backtest_signal_flip_asymmetric(proba, ret, thr_lo, thr_hi, cr=0.001, sess=None):
    """Асимметричный HOLD band: BUY при proba>=thr_hi, SELL при proba<=thr_lo, HOLD при thr_lo < proba < thr_hi."""
    pred = np.where(proba >= thr_hi, 1, np.where(proba <= thr_lo, 0, -1))
    n, pos, prev = len(proba), np.zeros(len(proba)), 0.0
    for i in range(n):
        if sess is not None and i and sess[i] != sess[i-1]: prev = 0.0
        if pred[i]==1: pos[i], prev = 1.0, 1.0
        elif pred[i]==0: pos[i], prev = -1.0, -1.0
        else: pos[i] = prev
    if sess is not None:
        last = np.zeros(n, dtype=bool); last[:-1] = sess[1:] != sess[:-1]; last[-1] = True
        pos = np.where(last & (pos != 0), 0.0, pos)
    return _bt(pos, ret, sess, cr)

def backtest_signal_flip(proba, ret, cr=0.001, thr=0.5, sess=None):
    pred = np.where(proba >= thr, 1, np.where(proba <= 1-thr, 0, -1))
    n, pos, prev = len(proba), np.zeros(len(proba)), 0.0
    for i in range(n):
        if sess is not None and i and sess[i] != sess[i-1]: prev = 0.0
        if pred[i]==1: pos[i], prev = 1.0, 1.0
        elif pred[i]==0: pos[i], prev = -1.0, -1.0
        else: pos[i] = prev
    if sess is not None:
        last = np.zeros(n, dtype=bool); last[:-1] = sess[1:] != sess[:-1]; last[-1] = True
        pos = np.where(last & (pos != 0), 0.0, pos)
    return _bt(pos, ret, sess, cr)

def backtest_pnl(proba, ret, commission_rt=0.001, threshold=0.5, session_ids=None, hold_keep_prev=True):
    """pred=1 -> long, pred=0 -> short, -1 -> HOLD.
    hold_keep_prev: при HOLD сохраняем прошлую позицию (меньше сделок). Иначе HOLD = flat.
    Комиссия 0.1% round-trip (0.05% в одну сторону)."""
    if threshold is None:
        raw = (proba >= 0.5).astype(int)
        pred = np.where(raw == 1, 1, 0)  # no HOLD
    else:
        pred = np.where(proba >= threshold, 1, np.where(proba <= 1 - threshold, 0, -1))
    
    n = len(proba)
    pos = np.zeros(n, dtype=np.float64)
    prev = 0.0
    for i in range(n):
        if session_ids is not None and i > 0 and session_ids[i] != session_ids[i - 1]:
            prev = 0.0
        if pred[i] == 1:
            pos[i] = 1.0
            prev = 1.0
        elif pred[i] == 0:
            pos[i] = -1.0
            prev = -1.0
        else:
            pos[i] = prev if hold_keep_prev else 0.0
    
    pos_prev = np.roll(pos, 1)
    pos_prev[0] = 0.0
    if session_ids is not None:
        sess_chg = np.zeros(n, dtype=bool)
        sess_chg[1:] = (session_ids[1:] != session_ids[:-1])
        pos_prev = np.where(sess_chg, 0.0, pos_prev)
    
    pos_changed = (pos != pos_prev) & ((pos != 0) | (pos_prev != 0))
    is_flip = pos_changed & (pos != 0) & (pos_prev != 0) & (pos * pos_prev < 0)
    fee_total = np.where(pos_changed, np.where(is_flip, commission_rt, commission_rt / 2), 0.0).sum()
    n_changes = int(pos_changed.sum())

    pnl_raw = pos * ret
    gross_pct = pnl_raw.sum() * 100
    net_pct = (pnl_raw.sum() - fee_total) * 100
    avg_gross = gross_pct / n_changes if n_changes > 0 else np.nan
    avg_net = net_pct / n_changes if n_changes > 0 else np.nan

    return {'trades': n_changes, 'gross_%': gross_pct, 'net_%': net_pct,
            'avg_gross_per_trade': avg_gross, 'avg_net_per_trade': avg_net}

## 3. Стратегии

**Одна модель, один день** — все результаты ниже по **последнему тестовому дню** (те же данные, что в шаге 1).

**Первая часть вывода** — два способа закрытия позиции (не две модели):
- **exit_at_05 (HOLD = flat):** при переходе proba в зону неопределённости (через 0.5) **выходим в кэш** → много мелких сделок, ниже net% и avg_net.
- **signal_flip (HOLD = keep):** при HOLD **держим позицию**; переворот только по противоположному сигналу; в конце сессии принудительно закрываем. Меньше сделок, выше net% и avg_net.

По каждому способу прогоняются пороги **thr** от 0.45 до 0.90 с шагом 0.05 (BUY при proba≥thr, SELL при ≤1-thr). HOLD band: (1-thr, thr), напр. thr=0.80 → band [0.20, 0.80]. **Oracle** — идеальный сигнал по target (look-ahead), верхняя граница.

**Вторая часть вывода** («Backtest на последнем дне…»)— тот же день и та же модель, но старый формат: одна логика (как signal_flip), разные варианты порога (каждый бар, tuned, wide HOLD 0.45–0.55 и т.д.). Сводная таблица в шаге 4 построена по **первой части** (exit_at_05 и signal_flip).

In [7]:
ret = bt['ret_next'].values
sess = bt['session_key'].values
target_signal = bt[TARGET_COL].values
THRESHOLDS = [0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90]  # шаг 0.05, от 0.45 до 0.90

results_exit = {}
results_flip = {}
for thr in THRESHOLDS:
    results_exit[thr] = backtest_exit_at_05(proba, ret, cr=COMMISSION_RT, thr=thr, sess=sess)
    results_flip[thr] = backtest_signal_flip(proba, ret, cr=COMMISSION_RT, thr=thr, sess=sess)
r_oracle_exit = backtest_exit_at_05((target_signal == 1).astype(float), ret, cr=COMMISSION_RT, thr=0.5, sess=sess)
r_oracle_flip = backtest_signal_flip((target_signal == 1).astype(float), ret, cr=COMMISSION_RT, thr=0.5, sess=sess)

print('=== exit_at_05 (HOLD = flat) ===')
for thr in THRESHOLDS:
    r = results_exit[thr]; an = r['avg_net_per_trade']
    s = f'{an:.2f}%' if not np.isnan(an) else 'N/A'
    print(f'  thr={thr:.2f}  trades={r["trades"]:>6,}  net={r["net_%"]:>8.2f}%  avg_net={s}')
print(f'  Oracle   trades={r_oracle_exit["trades"]:>6,}  net={r_oracle_exit["net_%"]:>8.2f}%')
print('=== signal_flip (HOLD=keep, close at session end) ===')
for thr in THRESHOLDS:
    r = results_flip[thr]; an = r['avg_net_per_trade']
    s = f'{an:.2f}%' if not np.isnan(an) else 'N/A'
    print(f'  thr={thr:.2f}  trades={r["trades"]:>6,}  net={r["net_%"]:>8.2f}%  avg_net={s}')
print(f'  Oracle   trades={r_oracle_flip["trades"]:>6,}  net={r_oracle_flip["net_%"]:>8.2f}%')

r1 = backtest_pnl(proba, ret, commission_rt=COMMISSION_RT, threshold=None, session_ids=sess)
r2 = backtest_pnl(proba, ret, commission_rt=COMMISSION_RT, threshold=BEST_THRESHOLD, session_ids=sess)
r_45 = backtest_pnl(proba, ret, commission_rt=COMMISSION_RT, threshold=0.45, session_ids=sess)
r_50 = backtest_pnl(proba, ret, commission_rt=COMMISSION_RT, threshold=0.50, session_ids=sess)
r3 = backtest_pnl(proba, ret, commission_rt=COMMISSION_RT, threshold=0.55, session_ids=sess)
r4 = backtest_pnl(proba, ret, commission_rt=COMMISSION_RT, threshold=0.60, session_ids=sess)  # широкая HOLD [0.40, 0.60]
r10 = backtest_pnl(proba, ret, commission_rt=COMMISSION_RT, threshold=0.65, session_ids=sess)
r5 = backtest_pnl(proba, ret, commission_rt=COMMISSION_RT, threshold=0.70, session_ids=sess)
r6 = backtest_pnl(proba, ret, commission_rt=COMMISSION_RT, threshold=0.75, session_ids=sess)
r7 = backtest_pnl(proba, ret, commission_rt=COMMISSION_RT, threshold=0.80, session_ids=sess)
r8 = backtest_pnl(proba, ret, commission_rt=COMMISSION_RT, threshold=0.85, session_ids=sess)
r9 = backtest_pnl(proba, ret, commission_rt=COMMISSION_RT, threshold=0.90, session_ids=sess)

target_signal = bt[TARGET_COL].values
r_oracle = backtest_pnl((target_signal == 1).astype(float), ret, commission_rt=COMMISSION_RT, threshold=None, session_ids=sess)

print('Backtest на последнем дне (комиссия 0.1% round-trip). HOLD = сохраняем позицию.')
for name, r in [('Модель (каждый бар)', r1), (f'Модель (tuned thr={BEST_THRESHOLD:.2f})', r2),
                ('Модель (thr 0.45)', r_45), ('Модель (thr 0.50)', r_50),
                ('Модель (wide HOLD 0.45-0.55)', r3), ('Модель (wide HOLD 0.40-0.60)', r4), ('Модель (wide HOLD 0.35-0.65)', r10),
                ('Модель (wide HOLD 0.30-0.70)', r5), ('Модель (wide HOLD 0.25-0.75)', r6), 
                ('Модель (wide HOLD 0.20-0.80)', r7), ('Модель (wide HOLD 0.15-0.85)', r8), 
                ('Модель (wide HOLD 0.10-0.90)', r9), ('Oracle (look-ahead)', r_oracle)]:
    avg_net = r["avg_net_per_trade"]
    avg_str = f'{avg_net:.2f}%' if not np.isnan(avg_net) else 'N/A'
    print(f'  {name:35} смен={r["trades"]:>6,}  gross={r["gross_%"]:>8.2f}%  net={r["net_%"]:>8.2f}%  avg_net={avg_str:>8}')

# Сводные таблицы: приоритет — avg_net_per_trade (прибыль на 1 сделку)
rows_all = []
for thr in THRESHOLDS:
    re, rf = results_exit[thr], results_flip[thr]
    rows_all.extend([{'mode': 'exit_at_05', 'thr': thr, **re}, {'mode': 'signal_flip', 'thr': thr, **rf}])
rows_all.extend([{'mode': 'exit_at_05', 'thr': 'oracle', **r_oracle_exit}, {'mode': 'signal_flip', 'thr': 'oracle', **r_oracle_flip}])
summary_all = pd.DataFrame(rows_all).sort_values('avg_net_per_trade', ascending=False)
summary_model = summary_all[summary_all['thr'] != 'oracle'].copy()

print('\n--- Сводная таблица (все режимы + oracle), сортировка по avg_net_per_trade ---')
display(summary_all.round({'gross_%': 2, 'net_%': 2, 'avg_net_per_trade': 4}))

print('\n--- Только модель (без oracle), сортировка по avg_net_per_trade ---')
display(summary_model.round({'gross_%': 2, 'net_%': 2, 'avg_net_per_trade': 4}))

# Сравнение способов закрытия при одних и тех же порогах
compare_thr = [0.45, 0.50, 0.60, 0.70, 0.80]  # включая новые пороги и продовый 0.8
compare_rows = []
for thr in compare_thr:
    compare_rows.append({**results_exit[thr], 'mode': 'exit_at_05', 'thr': thr})
    compare_rows.append({**results_flip[thr], 'mode': 'signal_flip', 'thr': thr})
compare_df = pd.DataFrame(compare_rows)[['mode', 'thr', 'trades', 'net_%', 'avg_net_per_trade']]
print('\n--- Сравнение exit_at_05 vs signal_flip (одни и те же пороги) ---')
display(compare_df.round({'net_%': 2, 'avg_net_per_trade': 4}))

=== exit_at_05 (HOLD = flat) ===
  thr=0.45  trades= 7,848  net=  993.64%  avg_net=0.13%
  thr=0.50  trades= 8,521  net=  961.84%  avg_net=0.11%
  thr=0.55  trades=12,506  net=  702.26%  avg_net=0.06%
  thr=0.60  trades=12,055  net=  509.69%  avg_net=0.04%
  thr=0.65  trades= 9,071  net=  418.76%  avg_net=0.05%
  thr=0.70  trades= 5,916  net=  304.65%  avg_net=0.05%
  thr=0.75  trades= 3,460  net=  210.49%  avg_net=0.06%
  thr=0.80  trades= 1,670  net=  120.10%  avg_net=0.07%
  thr=0.85  trades=   691  net=   73.49%  avg_net=0.11%
  thr=0.90  trades=   212  net=   26.28%  avg_net=0.12%
  Oracle   trades= 4,298  net= 4457.44%
=== signal_flip (HOLD=keep, close at session end) ===
  thr=0.45  trades= 7,935  net=  983.79%  avg_net=0.12%
  thr=0.50  trades= 8,604  net=  949.78%  avg_net=0.11%
  thr=0.55  trades= 5,091  net= 1055.95%  avg_net=0.21%
  thr=0.60  trades= 2,987  net= 1012.71%  avg_net=0.34%
  thr=0.65  trades= 1,849  net=  886.99%  avg_net=0.48%
  thr=0.70  trades= 1,199  net=  

,mode,thr,trades,gross_%,net_%,avg_net_per_trade
17,signal_flip,0.85,291,381.32,366.77,1.2604
15,signal_flip,0.8,457,529.48,506.63,1.1086
20,exit_at_05,oracle,4298,4672.34,4457.44,1.0371
21,signal_flip,oracle,4389,4648.53,4429.08,1.0091
19,signal_flip,0.9,142,128.15,121.05,0.8525
13,signal_flip,0.75,771,629.29,590.74,0.7662
11,signal_flip,0.7,1199,829.66,769.71,0.6420
9,signal_flip,0.65,1849,979.44,886.99,0.4797
7,signal_flip,0.6,2987,1162.06,1012.71,0.3390
5,signal_flip,0.55,5091,1310.50,1055.95,0.2074



--- Только модель (без oracle), сортировка по avg_net_per_trade ---


,mode,thr,trades,gross_%,net_%,avg_net_per_trade
17,signal_flip,0.85,291,381.32,366.77,1.2604
15,signal_flip,0.8,457,529.48,506.63,1.1086
19,signal_flip,0.9,142,128.15,121.05,0.8525
13,signal_flip,0.75,771,629.29,590.74,0.7662
11,signal_flip,0.7,1199,829.66,769.71,0.6420
9,signal_flip,0.65,1849,979.44,886.99,0.4797
7,signal_flip,0.6,2987,1162.06,1012.71,0.3390
5,signal_flip,0.55,5091,1310.50,1055.95,0.2074
0,exit_at_05,0.45,7848,1386.04,993.64,0.1266
1,signal_flip,0.45,7935,1380.54,983.79,0.1240



--- Сравнение exit_at_05 vs signal_flip (одни и те же пороги) ---


,mode,thr,trades,net_%,avg_net_per_trade
0,exit_at_05,0.45,7848,993.64,0.1266
1,signal_flip,0.45,7935,983.79,0.1240
2,exit_at_05,0.50,8521,961.84,0.1129
3,signal_flip,0.50,8604,949.78,0.1104
4,exit_at_05,0.60,12055,509.69,0.0423
5,signal_flip,0.60,2987,1012.71,0.3390
6,exit_at_05,0.70,5916,304.65,0.0515
7,signal_flip,0.70,1199,769.71,0.6420
8,exit_at_05,0.80,1670,120.10,0.0719
9,signal_flip,0.80,457,506.63,1.1086


## 4. Сводная таблица

Таблица собрана из **первой части** блока 3: строки — комбинации **mode** (exit_at_05 / signal_flip) и **thr** (порог 0.45–0.90, шаг 0.05). Сортировка по **avg_net_per_trade** (прибыль на 1 сделку) по убыванию — см. вывод в ячейке выше.

**Термины:**
- **gross_%** — суммарная доходность от сделок до вычета комиссий (pos × ret)
- **net_%** — доходность после комиссий (gross − fees), фактическая прибыль
- **avg_net_per_trade** — средняя прибыль на сделку (после комиссий), **основной приоритет** для выбора конфигурации
- **trades** — число смен позиции (каждая = 0.05% комиссии one-way). Ориентир: желательно 100–500 сделок (для интерпретации).

In [8]:
# Дублируем сводку для печати (summary_model и summary_all уже построены в ячейке выше)
summary = summary_all.copy()
print('Сводная таблица (все, по avg_net_per_trade):')
print(summary.round({'gross_%': 2, 'net_%': 2, 'avg_net_per_trade': 4}).to_string(index=False))

# Топ конфигураций по прибыли на сделку с пометкой диапазона сделок (ориентир 100–500)
def trades_range_label(trades):
    if trades < 100:
        return '<100'
    if trades <= 500:
        return '100–500'
    return '>500'

top_n = 12
top_df = summary_model.nlargest(top_n, 'avg_net_per_trade').copy()
top_df['trades_range'] = top_df['trades'].apply(trades_range_label)

from IPython.display import display, Markdown
md_lines = [
    '### Топ конфигураций по avg_net_per_trade (прибыль на 1 сделку)',
    '',
    '| mode | thr | trades | trades_range | net_% | avg_net_per_trade |',
    '|------|-----|--------|--------------|-------|-------------------|'
]
for _, row in top_df.iterrows():
    an = row['avg_net_per_trade']
    an_str = f"{an:.2f}%" if not pd.isna(an) else "N/A"
    md_lines.append(f"| {row['mode']} | {row['thr']} | {int(row['trades']):,} | {row['trades_range']} | {row['net_%']:.2f}% | {an_str} |")
display(Markdown('\n'.join(md_lines)))

Сводная таблица (все, по avg_net_per_trade):
       mode    thr  trades  gross_%   net_%  avg_net_per_trade
signal_flip   0.85     291   381.32  366.77             1.2604
signal_flip    0.8     457   529.48  506.63             1.1086
 exit_at_05 oracle    4298  4672.34 4457.44             1.0371
signal_flip oracle    4389  4648.53 4429.08             1.0091
signal_flip    0.9     142   128.15  121.05             0.8525
signal_flip   0.75     771   629.29  590.74             0.7662
signal_flip    0.7    1199   829.66  769.71             0.6420
signal_flip   0.65    1849   979.44  886.99             0.4797
signal_flip    0.6    2987  1162.06 1012.71             0.3390
signal_flip   0.55    5091  1310.50 1055.95             0.2074
 exit_at_05   0.45    7848  1386.04  993.64             0.1266
signal_flip   0.45    7935  1380.54  983.79             0.1240
 exit_at_05    0.9     212    36.88   26.28             0.1240
 exit_at_05    0.5    8521  1387.89  961.84             0.1129
signal_fli

### Топ конфигураций по avg_net_per_trade (прибыль на 1 сделку)

| mode | thr | trades | trades_range | net_% | avg_net_per_trade |
|------|-----|--------|--------------|-------|-------------------|
| signal_flip | 0.85 | 291 | 100–500 | 366.77% | 1.26% |
| signal_flip | 0.8 | 457 | 100–500 | 506.63% | 1.11% |
| signal_flip | 0.9 | 142 | 100–500 | 121.05% | 0.85% |
| signal_flip | 0.75 | 771 | >500 | 590.74% | 0.77% |
| signal_flip | 0.7 | 1,199 | >500 | 769.71% | 0.64% |
| signal_flip | 0.65 | 1,849 | >500 | 886.99% | 0.48% |
| signal_flip | 0.6 | 2,987 | >500 | 1012.71% | 0.34% |
| signal_flip | 0.55 | 5,091 | >500 | 1055.95% | 0.21% |
| exit_at_05 | 0.45 | 7,848 | >500 | 993.64% | 0.13% |
| signal_flip | 0.45 | 7,935 | >500 | 983.79% | 0.12% |
| exit_at_05 | 0.9 | 212 | 100–500 | 26.28% | 0.12% |
| exit_at_05 | 0.5 | 8,521 | >500 | 961.84% | 0.11% |

## 4.1 Эксперимент: асимметричные HOLD band'ы

Проверка асимметричных границ: разная чувствительность к BUY и SELL (напр. thr_lo=0.15, thr_hi=0.8 → band [0.15, 0.8]). **Эксперимент на одном тестовом дне** — для прода нужна валидация на других днях.

In [9]:
# Асимметричные band'ы (thr_lo, thr_hi): BUY>=thr_hi, SELL<=thr_lo, HOLD между
asym_bands = [
    (0.15, 0.80), (0.15, 0.85), (0.20, 0.80), (0.20, 0.85),
    (0.15, 0.75), (0.25, 0.80), (0.25, 0.85),
]
asym_rows = []
for thr_lo, thr_hi in asym_bands:
    r = backtest_signal_flip_asymmetric(proba, ret, thr_lo, thr_hi, cr=COMMISSION_RT, sess=sess)
    asym_rows.append({'band': f'{thr_lo}-{thr_hi}', 'thr_lo': thr_lo, 'thr_hi': thr_hi, **r})
asym_df = pd.DataFrame(asym_rows)
asym_df = asym_df.sort_values('avg_net_per_trade', ascending=False).reset_index(drop=True)
print('Асимметричные band\'ы (signal_flip), сортировка по avg_net_per_trade:')
display(asym_df.round({'gross_%': 2, 'net_%': 2, 'avg_net_per_trade': 4}))

# Сравнение с лучшими симметричными
print('Для сравнения — лучшие симметричные (thr 0.80, 0.85):')
for thr in [0.80, 0.85]:
    rs = results_flip[thr]
    print(f'  thr {thr} (band {1-thr:.2f}-{thr:.2f}): trades={rs["trades"]}, avg_net={rs["avg_net_per_trade"]:.4f}%')

Асимметричные band'ы (signal_flip), сортировка по avg_net_per_trade:


,band,thr_lo,thr_hi,trades,gross_%,net_%,avg_net_per_trade
0,0.15-0.8,0.15,0.80,351,490.71,473.16,1.3480
1,0.15-0.85,0.15,0.85,291,381.32,366.77,1.2604
2,0.15-0.75,0.15,0.75,391,469.06,449.51,1.1496
3,0.2-0.8,0.20,0.80,457,529.48,506.63,1.1086
4,0.2-0.85,0.20,0.85,352,337.24,319.64,0.9081
5,0.25-0.8,0.25,0.80,576,530.41,501.61,0.8708
6,0.25-0.85,0.25,0.85,400,352.43,332.43,0.8311


Для сравнения — лучшие симметричные (thr 0.80, 0.85):
  thr 0.8 (band 0.20-0.80): trades=457, avg_net=1.1086%
  thr 0.85 (band 0.15-0.85): trades=291, avg_net=1.2604%


## 5. Интерпретация и выводы

**Приоритет метрики:** **прибыль на 1 сделку (avg_net_per_trade)** — основной показатель. Ориентир по сделкам: **100–500** (не жёсткий фильтр).

### Что видно по результатам

**Симметричные band'ы (signal_flip):**
- **thr 0.85** (band 0.15–0.85): **1.26%** avg_net, 291 сделок — лидер по прибыли на сделку, сделок в целевом диапазоне.
- **thr 0.80** (band 0.20–0.80): **1.11%** avg_net, 457 сделок — второй результат, больше сделок и выше net_% (506%).
- При thr 0.90: 142 сделки, avg_net 0.85% — слишком мало сделок.
- При thr ≤0.75: avg_net падает (0.64–0.77%), сделок >500, комиссии заметнее.

**exit_at_05** (HOLD = flat) даёт на порядок меньше avg_net (0.04–0.13%) при большом числе сделок; по прибыли на сделку заметно уступает signal_flip.

**Асимметричные band'ы (эксперимент 4.1):**
- **0.15–0.80:** **1.35%** avg_net, 351 сделки — **лучший результат** среди проверенных, выше симметричного 0.85.
- 0.15–0.85: 1.26%, 291 — совпадает с симметричным thr 0.85.
- 0.2–0.80: 1.11%, 457 — совпадает с симметричным thr 0.80.

Сдвиг SELL-границы вниз (0.15 вместо 0.20) при той же BUY-границе 0.80 даёт прирост avg_net при достаточном числе сделок. **Требует проверки на других днях** — один тестовый день.

### Выводы

1. **Лучшая реалистичная конфигурация (симметричные):** signal_flip, thr 0.85 (1.26% avg_net, 291 сделок) или thr 0.80 (1.11%, 457 сделок).

2. **signal_flip** сильно выигрывает у **exit_at_05** по avg_net; удержание позиции при HOLD выгоднее, чем выход в кэш.

3. **Асимметричный band 0.15–0.80** дал максимальный avg_net (1.35%) — стоит прогнать на других днях перед продом.

4. **Oracle** — верхняя граница (look-ahead), в реале недостижим.

### Рекомендации

- **Максимум avg_net:** signal_flip, thr 0.85 или асимметричный 0.15–0.80 (после проверки на других днях).
- **Баланс avg_net и числа сделок:** signal_flip, thr 0.80 (457 сделок, 1.11% avg_net).
- **Продовый порог 0.80** (band 0.20–0.80) даёт стабильно хороший результат; 0.15–0.80 — кандидат для следующей итерации тестов.

## 6. Аудит пайплайна: откуда ~1000% за 1 день?

### Проверка на утечку данных

| Компонент | Статус | Пояснение |
|-----------|--------|-----------|
| **Фичи** | OK | Все фичи (`feature_pipeline.py`) используют только прошлые/текущие бары: `pct_change`, `diff`, `rolling`, `ewm`. Нет `shift(-1)`. Всё внутри `session_key` — границы сессий не пересекаются. |
| **Target** | OK | `tp_sl_1_05` — TP/SL разметка по **будущим** барам (H=20). Корреляция target с фичами слабая: `rd_regime: 0.064`, `rd_mom_1: 0.057`, `ret_1: -0.013`. Утечки нет. |
| **ret_next** | OK | `pct_change().shift(-1)` внутри `session_key` — доходность следующего бара. На момент решения (бар i) известна proba[i], зарабатываем ret[i→i+1]. Корректно. |
| **Temporal split** | OK | Train: 8 дней, Val: 1 день, Test: 1 день (последний). Модель не видела тестовый день. |
| **backtest_pnl** | OK | Позиция по proba текущего бара, PnL = pos × ret_next. HOLD сохраняет позицию. Комиссия при смене. |

### Почему ~1000%?

**Главная причина: суммирование по 60 параллельным инструментам.**

| Факт | Значение |
|------|----------|
| Баров на тестовом дне | 26 364 |
| Инструментов (символов) | **60** |
| Средних баров / символ | ~256 |
| Accuracy (угадано направление) | **53.5%** |
| Средний PnL на бар | 0.079% |
| Σ\|ret\| (максимально возможный gross) | 5 804% |
| Gross модели | 1 216% |
| Efficiency (model / max) | **21%** |
| Always long (baseline) | +120% |
| Random direction | -94% |

Бектест считает PnL **как будто мы одновременно торгуем все 60 символов на каждом баре**. Это означает:
- ~1000% — **не** доходность на один инструмент
- ~1000% / 60 символов ≈ **~17% на один символ за день** при ~256 барах
- Accuracy всего 53.5% — модель не «золотой грааль», но чуть лучше random
- Efficiency 21% от теоретического максимума

### Что не учтено (ограничения бектеста)

1. **Распределение капитала** — нельзя одновременно держать полную позицию по 60 инструментам с одним капиталом. В реальности нужно делить депозит.
2. **Проскальзывание** — не учтено. На мелких таймфреймах может съедать значительную часть прибыли.
3. **Ликвидность** — не все инструменты позволяют мгновенный вход/выход в нужном объёме.
4. **Один тестовый день** — результат может быть нерепрезентативным. Для надёжной оценки нужен walk-forward на нескольких днях.

### Итоговая оценка

Модель **работает лучше random** (random = -94%, always long = +120%, модель = +955–1002% net). Но абсолютные цифры завышены из-за суммирования по 60 инструментам. Реалистичная доходность на один инструмент — порядка **10–20% за день** на крипто, что само по себе высоко и требует подтверждения на walk-forward тестировании.